<a href="https://colab.research.google.com/github/hayatkhan20/umd-urban-heat-exposure/blob/main/notebooks/02_leaf_area_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip -q install -U earthengine-api geemap geopandas pyogrio

In [12]:
import ee
import geemap

ee.Authenticate()

PROJECT_ID = "aa-hayatnust"

ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


In [13]:
import os

repo_directory = "/content/umd-urban-heat-exposure"

if not os.path.exists(repo_directory):
    !git clone https://github.com/hayatkhan20/umd-urban-heat-exposure.git
else:
    print("Repository already exists.")

%cd /content/umd-urban-heat-exposure

Cloning into 'umd-urban-heat-exposure'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 188.00 KiB | 26.86 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/umd-urban-heat-exposure


In [14]:
import geopandas as gpd

boundary_path = "data/umd_boundary_final.geojson"

umd_boundary = gpd.read_file(boundary_path).to_crs("EPSG:4326")

print("Features:", len(umd_boundary))
print("Geometry:", umd_boundary.geometry.iloc[0].geom_type)
print("Valid:", umd_boundary.geometry.is_valid.iloc[0])
print("CRS:", umd_boundary.crs)

umd_ee = geemap.geopandas_to_ee(umd_boundary)

Features: 1
Geometry: Polygon
Valid: True
CRS: EPSG:4326


In [15]:
START_DATE = "2026-07-01"
END_DATE = "2026-08-29"

sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(umd_ee.geometry())
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

image_count = sentinel2.size().getInfo()

print("Sentinel-2 images found:", image_count)

Sentinel-2 images found: 13


In [16]:
scene_information = sentinel2.aggregate_array(
    "system:index"
).getInfo()

cloud_percentages = sentinel2.aggregate_array(
    "CLOUDY_PIXEL_PERCENTAGE"
).getInfo()

for scene, cloud in zip(scene_information, cloud_percentages):
    print(f"{scene} | scene cloud cover: {cloud:.2f}%")

20260723T154809_20260723T155507_T18SUJ | scene cloud cover: 0.26%
20260815T155819_20260815T160714_T18SUJ | scene cloud cover: 6.21%
20260713T160701_20260713T160704_T18SUJ | scene cloud cover: 8.11%
20260713T154809_20260713T155447_T18SUJ | scene cloud cover: 9.96%
20260703T154809_20260703T155539_T18SUJ | scene cloud cover: 14.28%
20260701T155821_20260701T160942_T18SUJ | scene cloud cover: 19.15%
20260809T155711_20260809T155805_T18SUJ | scene cloud cover: 22.47%
20260807T154811_20260807T155405_T18SUJ | scene cloud cover: 25.14%
20260825T155859_20260825T160759_T18SUJ | scene cloud cover: 39.03%
20260731T155821_20260731T160622_T18SUJ | scene cloud cover: 41.90%
20260720T155711_20260720T160051_T18SUJ | scene cloud cover: 42.08%
20260817T154811_20260817T155554_T18SUJ | scene cloud cover: 44.44%
20260726T155819_20260726T160923_T18SUJ | scene cloud cover: 51.44%


In [17]:
def mask_sentinel2_clouds(image):
    scl = image.select("SCL")

    clear_mask = (
        scl.neq(0)   # No data
        .And(scl.neq(1))   # Saturated/defective
        .And(scl.neq(3))   # Cloud shadow
        .And(scl.neq(8))   # Medium-probability cloud
        .And(scl.neq(9))   # High-probability cloud
        .And(scl.neq(10))  # Cirrus
        .And(scl.neq(11))  # Snow/ice
    )

    reflectance = (
        image.select(["B2", "B3", "B4", "B5", "B6",
                      "B7", "B8", "B8A", "B11", "B12"])
        .multiply(0.0001)
    )

    return (
        reflectance
        .updateMask(clear_mask)
        .copyProperties(image, image.propertyNames())
    )


clear_collection = sentinel2.map(mask_sentinel2_clouds)
summer_composite = clear_collection.median().clip(umd_ee.geometry())

Map = geemap.Map()
Map.centerObject(umd_ee, 14)

Map.addLayer(
    summer_composite,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0.02,
        "max": 0.30,
        "gamma": 1.2
    },
    "Sentinel-2 summer composite"
)

Map.addLayer(
    umd_ee.style(
        color="red",
        fillColor="00000000",
        width=3
    ),
    {},
    "UMD analysis boundary"
)

Map

Map(center=[38.989598798248835, -76.94166548680263], controls=(WidgetControl(options=['position', 'transparent…

In [18]:
import math
import ee

DEG_TO_RAD = math.pi / 180.0


def normalize(image, minimum, maximum):
    return (
        image.subtract(minimum)
        .multiply(2.0 / (maximum - minimum))
        .subtract(1)
    )


def tansig(image):
    return (
        image.multiply(-2)
        .exp()
        .add(1)
        .pow(-1)
        .multiply(2)
        .subtract(1)
    )


def linear_layer(inputs, weights, bias):
    result = ee.Image.constant(bias)

    for input_image, weight in zip(inputs, weights):
        result = result.add(input_image.multiply(weight))

    return result


def calculate_snap_lai(image):
    # Surface reflectance
    reflectance = image.select(
        ["B3", "B4", "B5", "B6", "B7", "B8A", "B11", "B12"]
    ).multiply(0.0001)

    b03 = normalize(reflectance.select("B3"), 0, 0.253061520471542)
    b04 = normalize(reflectance.select("B4"), 0, 0.290393577911328)
    b05 = normalize(reflectance.select("B5"), 0, 0.305398915248555)
    b06 = normalize(
        reflectance.select("B6"),
        0.006637972542253,
        0.608900395797889
    )
    b07 = normalize(
        reflectance.select("B7"),
        0.013972727018939,
        0.753827384322927
    )
    b8a = normalize(
        reflectance.select("B8A"),
        0.026690138082061,
        0.782011770669178
    )
    b11 = normalize(
        reflectance.select("B11"),
        0.016388074192258,
        0.493761397883092
    )
    b12 = normalize(
        reflectance.select("B12"),
        0,
        0.493025984460231
    )

    # Scene illumination and viewing angles
    view_zenith = ee.Number(
        image.get("MEAN_INCIDENCE_ZENITH_ANGLE_B8A")
    )
    view_azimuth = ee.Number(
        image.get("MEAN_INCIDENCE_AZIMUTH_ANGLE_B8A")
    )
    sun_zenith = ee.Number(
        image.get("MEAN_SOLAR_ZENITH_ANGLE")
    )
    sun_azimuth = ee.Number(
        image.get("MEAN_SOLAR_AZIMUTH_ANGLE")
    )

    view_cosine = ee.Image.constant(
        view_zenith.multiply(DEG_TO_RAD).cos()
    )
    sun_cosine = ee.Image.constant(
        sun_zenith.multiply(DEG_TO_RAD).cos()
    )
    relative_azimuth = ee.Image.constant(
        sun_azimuth
        .subtract(view_azimuth)
        .multiply(DEG_TO_RAD)
        .cos()
    )

    view_zenith_normalized = normalize(
        view_cosine,
        0.918595400582046,
        1
    )

    sun_zenith_normalized = normalize(
        sun_cosine,
        0.342022871159208,
        0.936206429175402
    )

    inputs = [
        b03,
        b04,
        b05,
        b06,
        b07,
        b8a,
        b11,
        b12,
        view_zenith_normalized,
        sun_zenith_normalized,
        relative_azimuth
    ]

    neuron1 = tansig(linear_layer(
        inputs,
        [
            -0.023406878966470,
             0.921655164636366,
             0.135576544080099,
            -1.938331472397950,
            -3.342495816122680,
             0.902277648009576,
             0.205363538258614,
            -0.040607844721716,
            -0.083196409727092,
             0.260029270773809,
             0.284761567218845
        ],
        4.96238030555279
    ))

    neuron2 = tansig(linear_layer(
        inputs,
        [
            -0.132555480856684,
            -0.139574837333540,
            -1.014606016898920,
            -1.330890038649270,
             0.031730624503341,
            -1.433583541317050,
            -0.959637898574699,
             1.133115706551000,
             0.216603876541632,
             0.410652303762839,
             0.064760155543506
        ],
        1.416008443981500
    ))

    neuron3 = tansig(linear_layer(
        inputs,
        [
             0.086015977724868,
             0.616648776881434,
             0.678003876446556,
             0.141102398644968,
            -0.096682206883546,
            -1.128832638862200,
             0.302189102741375,
             0.434494937299725,
            -0.021903699490589,
            -0.228492476802263,
            -0.039460537589826
        ],
        1.075897047213310
    ))

    neuron4 = tansig(linear_layer(
        inputs,
        [
            -0.109366593670404,
            -0.071046262972729,
             0.064582411478320,
             2.906325236823160,
            -0.673873108979163,
            -3.838051868280840,
             1.695979344531530,
             0.046950296081713,
            -0.049709652688365,
             0.021829545430994,
             0.057483827104091
        ],
        1.533988264655420
    ))

    neuron5 = tansig(linear_layer(
        inputs,
        [
            -0.089939416159969,
             0.175395483106147,
            -0.081847329172620,
             2.219895367487790,
             1.713873975136850,
             0.713069186099534,
             0.138970813499201,
            -0.060771761518025,
             0.124263341255473,
             0.210086140404351,
            -0.183878138700341
        ],
        3.024115930757230
    ))

    layer2 = (
        ee.Image.constant(1.096963107077220)
        .subtract(neuron1.multiply(1.500135489728730))
        .subtract(neuron2.multiply(0.096283269121503))
        .subtract(neuron3.multiply(0.194935930577094))
        .subtract(neuron4.multiply(0.352305895755591))
        .add(neuron5.multiply(0.075107415847473))
    )

    # Convert normalized model output into LAI
    lai = (
        layer2.add(1)
        .multiply(
            (14.4675094548151 - 0.000319182538301) / 2
        )
        .add(0.000319182538301)
        .rename("LAI")
    )

    scl = image.select("SCL")

    clear_mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    vegetation_mask = scl.eq(4)

    # Assign zero LAI to clear non-vegetated surfaces
    lai = lai.where(vegetation_mask.Not(), 0)

    # Retain clear pixels and remove implausible estimates
    lai = (
        lai.updateMask(clear_mask)
        .updateMask(lai.gte(0).And(lai.lte(8)))
        .clip(umd_ee.geometry())
    )

    return (
        lai.toFloat()
        .copyProperties(image, ["system:time_start", "system:index"])
    )

In [19]:
lai_collection = sentinel2.map(calculate_snap_lai)

print("LAI images processed:", lai_collection.size().getInfo())

umd_lai = (
    lai_collection
    .median()
    .rename("LAI")
    .clip(umd_ee.geometry())
)

valid_observations = (
    lai_collection
    .count()
    .rename("valid_observations")
    .clip(umd_ee.geometry())
)

lai_statistics = umd_lai.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(
            ee.Reducer.percentile([25, 50, 75, 95]),
            sharedInputs=True
        )
    ),
    geometry=umd_ee.geometry(),
    scale=20,
    crs="EPSG:26918",
    bestEffort=True,
    maxPixels=1e8
).getInfo()

print("LAI statistics:")
for key, value in lai_statistics.items():
    print(f"{key}: {value}")

LAI images processed: 13
LAI statistics:
LAI_max: 4.228519916534424
LAI_mean: 1.2807261677116688
LAI_min: 0
LAI_p25: 0
LAI_p50: 1.2963561057272117
LAI_p75: 2.203366113968219
LAI_p95: 2.9840179266271214


In [20]:
lai_style = {
    "min": 0,
    "max": 6,
    "palette": [
        "#f7f7f7",
        "#d9f0a3",
        "#addd8e",
        "#78c679",
        "#41ab5d",
        "#238443",
        "#005a32"
    ]
}

observation_style = {
    "min": 1,
    "max": 13,
    "palette": ["#fff7bc", "#fec44f", "#d95f0e"]
}

Map = geemap.Map()
Map.centerObject(umd_ee, 14)

Map.addLayer(
    summer_composite,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0.02,
        "max": 0.30,
        "gamma": 1.2
    },
    "Sentinel-2 composite",
    False
)

Map.addLayer(
    umd_lai,
    lai_style,
    "Sentinel-2 LAI"
)

Map.addLayer(
    valid_observations,
    observation_style,
    "Valid observation count",
    False
)

Map.addLayer(
    umd_ee.style(
        color="red",
        fillColor="00000000",
        width=3
    ),
    {},
    "UMD boundary"
)

Map.add_colorbar(
    lai_style,
    label="Leaf Area Index (m² leaf area / m² ground area)"
)

Map

Map(center=[38.989598798248835, -76.94166548680263], controls=(WidgetControl(options=['position', 'transparent…

In [21]:
import os
from google.colab import files

output_path = "/content/umd_lai_summer_2026_20m.tif"

geemap.ee_export_image(
    umd_lai,
    filename=output_path,
    scale=20,
    crs="EPSG:26918",
    region=umd_ee.geometry(),
    file_per_band=False
)

if os.path.exists(output_path):
    print("Created:", output_path)
    print(
        "File size:",
        round(os.path.getsize(output_path) / 1_000_000, 2),
        "MB"
    )
    files.download(output_path)
else:
    print("Export failed: output file was not created.")

Generating URL ...
Please wait ...
Data downloaded to /content/umd_lai_summer_2026_20m.tif
Created: /content/umd_lai_summer_2026_20m.tif
File size: 0.07 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>